In [2]:
import sys
print("Python:", sys.version)

%pip --version

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


In [1]:
import shutil

ffmpeg = shutil.which("ffmpeg")
print("FFmpeg:", ffmpeg or "NOT INSTALLED")

FFmpeg: /usr/bin/ffmpeg


In [2]:

import math
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import Video, display

# 1. Check FFmpeg
ffmpeg = shutil.which("ffmpeg")
if not ffmpeg:
    raise RuntimeError("FFmpeg was not found.")

# 2. Render settings
W, H = 640, 360
FPS = 120
DURATION = 3
TOTAL_FRAMES = FPS * DURATION
OUTPUT = "/content/motionforge_120fps.mp4"
LOG = "/content/motionforge_ffmpeg.log"

# 3. Start FFmpeg: accept raw RGB frames and encode MP4
cmd = [
    ffmpeg, "-y", "-loglevel", "error",
    "-f", "rawvideo",
    "-pixel_format", "rgb24",
    "-video_size", f"{W}x{H}",
    "-framerate", str(FPS),
    "-i", "-",
    "-an",
    "-c:v", "libx264",
    "-preset", "ultrafast",
    "-crf", "18",
    "-pix_fmt", "yuv420p",
    OUTPUT
]

with open(LOG, "w") as log:
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=log)

    try:
        for i in range(TOTAL_FRAMES):
            # Deterministic animation time
            t = i / FPS

            frame = np.zeros((H, W, 3), dtype=np.uint8)
            frame[:] = (11, 16, 32)

            img = Image.fromarray(frame)
            draw = ImageDraw.Draw(img)

            # Smooth circular movement
            x = int(W / 2 + 180 * math.sin(2 * math.pi * t / DURATION))
            y = int(H / 2 + 65 * math.cos(2 * math.pi * t / DURATION))

            # Glowing-style layered orb
            for r, color in [
                (44, (24, 65, 110)),
                (30, (30, 135, 205)),
                (17, (100, 225, 255))
            ]:
                draw.ellipse(
                    (x-r, y-r, x+r, y+r),
                    fill=color
                )

            draw.text(
                (20, 20),
                f"MOTIONFORGE  |  {t:.2f}s  |  {FPS} FPS",
                fill=(230, 240, 255)
            )

            # Stream this frame directly to FFmpeg
            proc.stdin.write(np.asarray(img).tobytes())

            if (i + 1) % 120 == 0:
                print(f"Rendered {i + 1}/{TOTAL_FRAMES} frames")

        proc.stdin.close()
        return_code = proc.wait()

    except Exception:
        proc.kill()
        proc.wait()
        raise

if return_code != 0:
    with open(LOG) as f:
        print(f.read())
    raise RuntimeError("FFmpeg export failed.")

print("✅ MotionForge render complete:", OUTPUT)
print("Frames:", TOTAL_FRAMES)
print("Frame rate:", FPS, "FPS")
print("Duration:", DURATION, "seconds")

display(Video(OUTPUT, embed=True))


Rendered 120/360 frames
Rendered 240/360 frames
Rendered 360/360 frames
✅ MotionForge render complete: /content/motionforge_120fps.mp4
Frames: 360
Frame rate: 120 FPS
Duration: 3 seconds


In [3]:
import shutil
import subprocess

ffmpeg = shutil.which("ffmpeg")

if ffmpeg:
    print("✅ FFmpeg is installed!")
    subprocess.run([ffmpeg, "-version"], check=True)
else:
    print("❌ FFmpeg is NOT installed.")

✅ FFmpeg is installed!


In [4]:
import os

print("Working directory:", os.getcwd())
print("Files:")
for item in os.listdir("/content"):
    print(" -", item)

Working directory: /content
Files:
 - .config
 - motionforge_ffmpeg.log
 - motionforge_120fps.mp4
 - sample_data


In [3]:
%pip install -v manim 2>&1


Streaming output truncated to the last 5000 lines.
    Skipping link: none of the wheel's tags (cp38-cp38-manylinux1_i686) are compatible (run pip debug --verbose to show compatible tags): https://files.pythonhosted.org/packages/4b/3a/e19fd1c0d2eaf1e783be5f99dc9c12811786f500e239766db7bc6ebab8b5/Cython-0.29.22-cp38-cp38-manylinux1_i686.whl (from https://pypi.org/simple/cython/) (requires-python:>=2.6, !=3.0.*, !=3.1.*, !=3.2.*)
    Skipping link: none of the wheel's tags (cp38-cp38-manylinux1_x86_64) are compatible (run pip debug --verbose to show compatible tags): https://files.pythonhosted.org/packages/f9/f4/3a950e0146569f405ac3ea1233dc08bb2e2286cb7356de866807fde94f39/Cython-0.29.22-cp38-cp38-manylinux1_x86_64.whl (from https://pypi.org/simple/cython/) (requires-python:>=2.6, !=3.0.*, !=3.1.*, !=3.2.*)
    Skipping link: none of the wheel's tags (cp38-cp38-win32) are compatible (run pip debug --verbose to show compatible tags): https://files.pythonhosted.org/packages/81/fb/83993400b0f

In [1]:
import sys
print("Python:", sys.version)

try:
    import manim
    print("Manim version:", manim.__version__)
    print("✅ MotionForge environment ready!")
except Exception as e:
    print("❌ Import failed:", repr(e))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
❌ Import failed: ModuleNotFoundError("No module named 'manim'")


In [3]:
!apt-get update -qq
!apt-get install -y -qq pkg-config libcairo2-dev \
  libpango1.0-dev ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libdatrie-dev:amd64.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../00-libdatrie-dev_0.2.13-3build1_amd64.deb ...
Unpacking libdatrie-dev:amd64 (0.2.13-3build1) ...
Selecting previously unselected package libfribidi-dev:amd64.
Preparing to unpack .../01-libfribidi-dev_1.0.13-3build1_amd64.deb ...
Unpacking libfribidi-dev:amd64 (1.0.13-3build1) ...
Selecting previously unselected package libgraphite2-dev:amd64.
Preparing to unpack .../02-libgraphite2-dev_1.3.14-2ubuntu0.24.04.1_amd64.deb ...
Unpacking libgraphite2-dev:amd64 (1.3.14-2ubuntu0.24.04.1) ...
Selecting previously unselected package libharfbuzz-cairo0:amd64.
Preparing to unpack .../03-libharfbuzz-cairo0_8.3.0-2build2_amd64.deb ...
Unpacking libharfbuzz-

In [4]:
%pip install manim

  Using cached manim-0.22.0-py3-none-any.whl.metadata (10 kB)
  Using cached av-19.0.2-cp312-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached cloup-4.0.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached decorator-5.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached isosurfaces-0.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached manimpango-0.7.0.tar.gz (4.1 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.7/700.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━